In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fpdf2 flask tensorflow pillow pyngrok werkzeug numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 21.4 MB/s eta 0:00:00


In [3]:
#!/usr/bin/env python3
"""
OncoScan AI — Production Flask Application
==========================================
Install: pip install flask tensorflow pillow pyngrok fpdf2 numpy werkzeug

Features:
  - VGG16-based lung cancer histology classifier
  - SQLite patient records database
  - Downloadable PDF clinical reports (fpdf2)
  - Real-time dashboard with Chart.js visualisations
  - Full confidence score breakdown per scan
"""

import os, sqlite3, json, uuid, io, base64
from datetime import datetime, timedelta
from flask import Flask, request, jsonify, send_file, g

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.layers import Layer
import tensorflow as tf
from PIL import Image
import numpy as np
from pyngrok import ngrok
from werkzeug.utils import secure_filename
from fpdf import FPDF

# ─────────────────────────── CONFIG ───────────────────────────────────────────
DATABASE   = '/content/drive/MyDrive/lungcancer/oncoscan.db'
MODEL_PATH = '/content/drive/MyDrive/lungcancer/saved_models/vgg16_model.keras'
NGROK_TOKEN = "2rInonMJZkVbZFBCLMHqOh4GK5n_3RuT7AHDPG2GdESffcGk5"   # <── replace
ALLOWED_EXT = {'png', 'jpg', 'jpeg'}

CLASS_LABELS = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal",
    "Squamous Cell Carcinoma",
]

CLASS_INFO = {
    "Adenocarcinoma": {
        "abbr": "ADC", "risk": "High", "hex": "#ef4444",
        "icd10": "C34.1",
        "description": (
            "Adenocarcinoma is the most common subtype of non-small-cell lung cancer (NSCLC), "
            "accounting for ~40% of all cases. It originates in peripheral glandular cells and "
            "exhibits glandular formations, mucin production, or papillary growth patterns. "
            "Growth is typically slower than other subtypes. Early-stage 5-year survival reaches "
            "57% for localised disease."
        ),
        "recommendation": (
            "Immediate referral to oncology. CT-guided biopsy for molecular profiling. "
            "EGFR, ALK, ROS1, KRAS mutation testing. Stage-dependent: surgical resection (I–II), "
            "targeted therapy or immunotherapy (III–IV)."
        ),
    },
    "Large Cell Carcinoma": {
        "abbr": "LCC", "risk": "High", "hex": "#f97316",
        "icd10": "C34.2",
        "description": (
            "Large Cell Carcinoma (LCC) is an undifferentiated NSCLC representing ~10–15% of lung "
            "cancers. It is characterised by large polygonal cells without squamous or glandular "
            "features. LCC grows and spreads rapidly, carries a higher metastatic potential, and "
            "is often peripherally located."
        ),
        "recommendation": (
            "Urgent oncology consultation. PET-CT for staging. PD-L1 testing for immunotherapy "
            "eligibility. Standard first-line: carboplatin/paclitaxel combination chemotherapy "
            "for advanced stages."
        ),
    },
    "Normal": {
        "abbr": "NRM", "risk": "Low", "hex": "#22c55e",
        "icd10": "Z03.89",
        "description": (
            "No malignant cellular characteristics detected. Sample shows regular alveolar "
            "architecture with intact bronchiolar epithelium. No nuclear atypia, abnormal mitotic "
            "figures, or tissue invasion observed. Cellular morphology is within expected "
            "physiological parameters."
        ),
        "recommendation": (
            "No malignancy detected. Annual low-dose CT recommended for high-risk patients "
            "(age >55, heavy smokers). Maintain preventive care schedule and lifestyle modifications "
            "to reduce future lung cancer risk."
        ),
    },
    "Squamous Cell Carcinoma": {
        "abbr": "SCC", "risk": "High", "hex": "#a855f7",
        "icd10": "C34.3",
        "description": (
            "Squamous Cell Carcinoma (SCC) accounts for 25–30% of lung cancers. It arises from "
            "squamous cells lining the bronchi and is strongly associated with tobacco smoking. "
            "Histologically characterised by keratin pearls and intercellular bridges. Typically "
            "presents as a central mass near main bronchi with local structural invasion."
        ),
        "recommendation": (
            "Bronchoscopic evaluation and biopsy confirmation. PD-L1 expression testing. "
            "Cisplatin-based chemo-radiation for unresectable disease. Surgical resection "
            "for early-stage candidates."
        ),
    },
}

# ─────────────────────────── CUSTOM LAYERS ────────────────────────────────────
class Cast(Layer):
    def call(self, inputs):
        return tf.cast(inputs, tf.float16)

class TFSMLayer(Layer):
    def __init__(self, **kwargs):
        kwargs.pop('filepath', None); kwargs.pop('call_endpoint', None)
        kwargs.pop('call_training_endpoint', None)
        super().__init__(**kwargs)
    def call(self, inputs): return inputs

# ─────────────────────────── MODEL LOAD ───────────────────────────────────────
print("[OncoScan AI] Loading VGG16 model …")
model = load_model(MODEL_PATH)
print("[OncoScan AI] Model ready.")

# ─────────────────────────── FLASK APP ────────────────────────────────────────
app = Flask(__name__)

# ─────────────────────────── DATABASE ─────────────────────────────────────────
def get_db():
    db = getattr(g, '_db', None)
    if db is None:
        db = g._db = sqlite3.connect(DATABASE)
        db.row_factory = sqlite3.Row
    return db

@app.teardown_appcontext
def close_db(_):
    db = getattr(g, '_db', None)
    if db: db.close()

def init_db():
    with app.app_context():
        get_db().executescript("""
            CREATE TABLE IF NOT EXISTS scans (
                id             TEXT PRIMARY KEY,
                patient_name   TEXT NOT NULL,
                patient_age    INTEGER,
                patient_gender TEXT,
                prediction     TEXT NOT NULL,
                confidence     REAL NOT NULL,
                all_scores     TEXT NOT NULL,
                image_b64      TEXT,
                created_at     TEXT NOT NULL
            );
        """)
        get_db().commit()

# ─────────────────────────── PDF GENERATOR ────────────────────────────────────
def generate_pdf(scan: dict) -> bytes:
    # 1. Fetch info
    info = CLASS_INFO[scan["prediction"]]
    scores = json.loads(scan["all_scores"])

    # 2. SANITIZE: Remove fancy dashes/quotes that cause the 'latin-1' error
    # This replaces long dashes (—) and en-dashes (–) with standard hyphens (-)
    def clean_text(t):
        return str(t).replace('\u2013', '-').replace('\u2014', '-').replace('~', 'approx. ')

    desc = clean_text(info["description"])
    rec = clean_text(info["recommendation"])
    patient_name = clean_text(scan['patient_name'])

    # 3. Initialize FPDF (fpdf2 version)
    pdf = FPDF()
    pdf.add_page()
    W = 210

    # Header
    pdf.set_fill_color(8, 13, 22)
    pdf.rect(0, 0, W, 38, "F")
    pdf.set_font("Helvetica", "B", 20)
    pdf.set_text_color(20, 184, 166)
    pdf.set_xy(12, 7)
    pdf.cell(0, 10, "ONCOSCAN AI", ln=True)

    # Patient Info Box
    pdf.set_y(43)
    pdf.set_fill_color(240, 246, 255)
    pdf.rect(10, 43, W - 20, 28, "F")
    pdf.set_font("Helvetica", "B", 11); pdf.set_text_color(20, 30, 60)
    pdf.set_xy(14, 46)
    pdf.cell(90, 7, f"Patient: {patient_name}", ln=False)
    pdf.cell(90, 7, f"ICD-10: {info['icd10']}", ln=True)

    # Diagnosis Banner
    pdf.set_y(76)
    r, g2, b = (239, 68, 68) if info["risk"] == "High" else (34, 197, 94)
    pdf.set_fill_color(r, g2, b)
    pdf.rect(10, 76, W - 20, 16, "F")
    pdf.set_font("Helvetica", "B", 13); pdf.set_text_color(255, 255, 255)
    pdf.set_xy(14, 80)
    pdf.cell(120, 8, f"DIAGNOSIS: {scan['prediction'].upper()}", ln=False)
    pdf.cell(60, 8, f"Confidence: {float(scan['confidence']):.1f}%", ln=True, align="R")

    # Clinical Text (Using the cleaned variables)
    pdf.set_y(pdf.get_y() + 10)
    pdf.set_font("Helvetica", "B", 10); pdf.set_text_color(20, 30, 60)
    pdf.cell(0, 8, "CLINICAL DESCRIPTION", ln=True)
    pdf.set_font("Helvetica", "", 9); pdf.set_text_color(60, 70, 100)
    pdf.set_x(14); pdf.multi_cell(W - 28, 5, desc)

    pdf.set_y(pdf.get_y() + 5)
    pdf.set_font("Helvetica", "B", 10); pdf.set_text_color(20, 30, 60)
    pdf.cell(0, 8, "CLINICAL RECOMMENDATIONS", ln=True)
    pdf.set_font("Helvetica", "", 9); pdf.set_text_color(60, 70, 100)
    pdf.set_x(14); pdf.multi_cell(W - 28, 5, rec)

    # Footer
    pdf.set_y(277)
    pdf.set_font("Helvetica", "", 7); pdf.set_text_color(150, 155, 170)
    pdf.cell(0, 5, "OncoScan AI Clinical Record", align="C")

    # Return the byte stream directly
    return pdf.output()

# ─────────────────────────── HELPERS ──────────────────────────────────────────
def allowed_file(fn): return '.' in fn and fn.rsplit('.', 1)[1].lower() in ALLOWED_EXT

# ─────────────────────────── ROUTES ───────────────────────────────────────────
@app.route("/")
def index(): return HTML_TEMPLATE

@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        return jsonify({"error": "No file in request"}), 400
    file = request.files["file"]
    if not file.filename or not allowed_file(file.filename):
        return jsonify({"error": "Invalid file. Upload JPG/PNG only."}), 400

    patient_name   = request.form.get("patient_name", "Anonymous").strip() or "Anonymous"
    patient_age    = request.form.get("patient_age", "N/A")
    patient_gender = request.form.get("patient_gender", "Unspecified")

    try:
        raw = file.read()
        img = Image.open(io.BytesIO(raw)).convert("RGB").resize((224, 224))
        arr = np.expand_dims(img_to_array(img) / 255.0, axis=0)
        preds = model.predict(arr)[0]
        idx   = int(np.argmax(preds))
        label = CLASS_LABELS[idx]
        conf  = float(preds[idx]) * 100
        img_b64 = base64.b64encode(raw).decode("utf-8")

        scan_id = str(uuid.uuid4())
        now     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        db = get_db()
        db.execute(
            "INSERT INTO scans VALUES (?,?,?,?,?,?,?,?,?)",
            (scan_id, patient_name, patient_age, patient_gender,
             label, conf, json.dumps(preds.tolist()), img_b64, now)
        )
        db.commit()

        return jsonify({
            "scan_id":      scan_id,
            "prediction":   label,
            "confidence":   f"{conf:.2f}",
            "all_scores":   {CLASS_LABELS[i]: f"{float(preds[i])*100:.2f}" for i in range(4)},
            "risk":         CLASS_INFO[label]["risk"],
            "hex":          CLASS_INFO[label]["hex"],
            "description":  CLASS_INFO[label]["description"],
            "recommendation": CLASS_INFO[label]["recommendation"],
            "icd10":        CLASS_INFO[label]["icd10"],
            "created_at":   now,
        })
    except Exception as e:
        return jsonify({"error": f"Processing failed: {str(e)}"}), 500

@app.route("/api/stats")
def api_stats():
    db = get_db()
    total = db.execute("SELECT COUNT(*) FROM scans").fetchone()[0]
    cancer_ct = db.execute(
        "SELECT COUNT(*) FROM scans WHERE prediction != 'Normal'"
    ).fetchone()[0]
    avg_conf = db.execute("SELECT AVG(confidence) FROM scans").fetchone()[0] or 0
    today = datetime.now().strftime("%Y-%m-%d")
    today_ct = db.execute(
        "SELECT COUNT(*) FROM scans WHERE created_at LIKE ?", (today + "%",)
    ).fetchone()[0]

    # Distribution
    dist = {}
    for row in db.execute("SELECT prediction, COUNT(*) as cnt FROM scans GROUP BY prediction"):
        dist[row["prediction"]] = row["cnt"]

    # Last 7 days trend
    trend = []
    for i in range(6, -1, -1):
        day = (datetime.now() - timedelta(days=i)).strftime("%Y-%m-%d")
        cnt = db.execute(
            "SELECT COUNT(*) FROM scans WHERE created_at LIKE ?", (day + "%",)
        ).fetchone()[0]
        trend.append({"date": day, "count": cnt})

    # Avg confidence per class
    conf_per_class = {}
    for row in db.execute(
        "SELECT prediction, AVG(confidence) as ac FROM scans GROUP BY prediction"
    ):
        conf_per_class[row["prediction"]] = round(row["ac"], 1)

    return jsonify({
        "total": total, "cancer_positive": cancer_ct,
        "avg_confidence": round(avg_conf, 1), "today": today_ct,
        "distribution": dist, "trend": trend,
        "conf_per_class": conf_per_class,
    })

@app.route("/api/history")
def api_history():
    rows = get_db().execute(
        "SELECT id, patient_name, patient_age, patient_gender, "
        "prediction, confidence, created_at FROM scans ORDER BY created_at DESC LIMIT 100"
    ).fetchall()
    return jsonify([dict(r) for r in rows])

@app.route("/report/pdf/<scan_id>")
def download_pdf(scan_id):
    row = get_db().execute("SELECT * FROM scans WHERE id=?", (scan_id,)).fetchone()
    if not row:
        return jsonify({"error": "Scan not found"}), 404
    pdf_bytes = generate_pdf(dict(row))
    buf = io.BytesIO(pdf_bytes)
    buf.seek(0)
    return send_file(buf, mimetype="application/pdf",
                     as_attachment=True,
                     download_name=f"OncoScan_{scan_id[:8]}.pdf")

# ─────────────────────────── HTML TEMPLATE ────────────────────────────────────
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>OncoScan AI — Lung Cancer Detection Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@600;800&family=Exo+2:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
:root{
  --bg:#080d14;--card:#0d1526;--card2:#111d35;
  --border:#1e3050;--border2:#243d60;
  --teal:#14b8a6;--teal2:#0d9488;--teal-glow:rgba(20,184,166,.15);
  --red:#ef4444;--green:#22c55e;--amber:#f59e0b;--purple:#a855f7;
  --text:#e2eaf8;--text2:#8aa4c8;--text3:#4a6485;
  --font:'Exo 2',sans-serif;--mono:'JetBrains Mono',monospace;
  --r:14px;--r2:8px;
}
*{margin:0;padding:0;box-sizing:border-box}
html,body{height:100%;background:var(--bg);color:var(--text);font-family:var(--font)}
::-webkit-scrollbar{width:6px;background:var(--bg)}
::-webkit-scrollbar-thumb{background:var(--border2);border-radius:3px}

/* ── Layout ── */
.shell{display:flex;height:100vh;overflow:hidden}
.sidebar{width:240px;min-width:240px;background:var(--card);border-right:1px solid var(--border);
  display:flex;flex-direction:column;padding:0 0 20px}
.main{flex:1;overflow-y:auto;background:var(--bg)}

/* ── Sidebar ── */
.logo{padding:26px 22px 20px;border-bottom:1px solid var(--border)}
.logo-title{font-family:'Orbitron',sans-serif;font-size:1.15rem;font-weight:800;
  background:linear-gradient(90deg,var(--teal),#67e8f9);-webkit-background-clip:text;
  -webkit-text-fill-color:transparent}
.logo-sub{font-size:.7rem;color:var(--text3);margin-top:3px;letter-spacing:.08em}
.nav{padding:16px 12px;flex:1}
.nav-item{display:flex;align-items:center;gap:12px;padding:11px 14px;border-radius:10px;
  cursor:pointer;color:var(--text2);font-size:.87rem;font-weight:500;
  transition:all .2s;margin-bottom:4px;border:1px solid transparent}
.nav-item:hover{background:var(--card2);color:var(--text)}
.nav-item.active{background:var(--teal-glow);color:var(--teal);border-color:rgba(20,184,166,.3)}
.nav-icon{width:20px;text-align:center;font-size:1rem}
.sidebar-badge{margin-left:auto;background:var(--teal);color:#fff;font-size:.68rem;
  padding:2px 7px;border-radius:12px;font-family:var(--mono)}

/* ── Page header ── */
.page-header{padding:28px 32px 0}
.page-title{font-family:'Orbitron',sans-serif;font-size:1.3rem;font-weight:600;
  letter-spacing:.05em}
.page-sub{color:var(--text3);font-size:.82rem;margin-top:4px}

/* ── View sections ── */
.view{display:none;padding:24px 32px 40px}
.view.active{display:block}

/* ── Stat cards ── */
.stats-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:28px}
.stat-card{background:var(--card);border:1px solid var(--border);border-radius:var(--r);
  padding:20px;position:relative;overflow:hidden;transition:border-color .2s}
.stat-card:hover{border-color:var(--border2)}
.stat-card::before{content:'';position:absolute;top:0;left:0;right:0;height:3px}
.stat-card.teal::before{background:linear-gradient(90deg,var(--teal),#67e8f9)}
.stat-card.red::before{background:linear-gradient(90deg,var(--red),#fb923c)}
.stat-card.green::before{background:linear-gradient(90deg,var(--green),#86efac)}
.stat-card.purple::before{background:linear-gradient(90deg,var(--purple),#c084fc)}
.stat-label{font-size:.72rem;color:var(--text3);letter-spacing:.06em;text-transform:uppercase;margin-bottom:10px}
.stat-value{font-family:var(--mono);font-size:2rem;font-weight:600;line-height:1}
.stat-sub{font-size:.75rem;color:var(--text3);margin-top:6px}
.stat-icon{position:absolute;right:18px;top:18px;font-size:1.6rem;opacity:.15}

/* ── Chart cards ── */
.charts-row{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:28px}
.chart-card{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:22px}
.chart-title{font-size:.8rem;font-weight:600;letter-spacing:.06em;text-transform:uppercase;
  color:var(--text2);margin-bottom:18px;display:flex;align-items:center;gap:8px}
.chart-title span{display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--teal)}
.chart-wrap{position:relative;height:220px}

/* ── Analyze layout ── */
.analyze-grid{display:grid;grid-template-columns:380px 1fr;gap:20px;align-items:start}
.panel{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:24px}
.panel-title{font-size:.78rem;font-weight:600;letter-spacing:.07em;text-transform:uppercase;
  color:var(--teal);margin-bottom:18px;display:flex;align-items:center;gap:8px}
.panel-title::before{content:'';display:inline-block;width:3px;height:14px;
  background:var(--teal);border-radius:2px}

/* ── Form ── */
.field{margin-bottom:16px}
.field label{display:block;font-size:.78rem;color:var(--text2);margin-bottom:6px;font-weight:500}
.field input,.field select{width:100%;background:var(--card2);border:1px solid var(--border2);
  border-radius:var(--r2);padding:10px 14px;color:var(--text);font-family:var(--font);
  font-size:.9rem;outline:none;transition:border-color .2s}
.field input:focus,.field select:focus{border-color:var(--teal)}
.field select option{background:var(--card2)}

/* ── Upload zone ── */
.upload-zone{border:2px dashed var(--border2);border-radius:var(--r);padding:28px;
  text-align:center;cursor:pointer;transition:all .25s;margin-top:4px;position:relative}
.upload-zone:hover,.upload-zone.drag{border-color:var(--teal);background:var(--teal-glow)}
.upload-zone input{position:absolute;inset:0;opacity:0;cursor:pointer;width:100%;height:100%}
.upload-icon{font-size:2.2rem;margin-bottom:10px;opacity:.6}
.upload-text{color:var(--text2);font-size:.88rem}
.upload-text strong{color:var(--teal)}
.preview-img{max-width:100%;border-radius:var(--r2);margin-top:14px;
  border:2px solid var(--border2);display:none}

/* ── Analyse button ── */
.btn-analyse{width:100%;margin-top:16px;padding:13px;border:none;border-radius:var(--r2);
  background:linear-gradient(135deg,var(--teal),var(--teal2));color:#fff;
  font-family:var(--font);font-size:.95rem;font-weight:600;cursor:pointer;
  transition:opacity .2s,transform .15s;letter-spacing:.04em}
.btn-analyse:hover:not(:disabled){opacity:.9;transform:translateY(-1px)}
.btn-analyse:disabled{opacity:.45;cursor:not-allowed}

/* ── Result panel ── */
#result-panel{display:none}
.diagnosis-banner{border-radius:var(--r2);padding:16px 20px;margin-bottom:20px;
  display:flex;align-items:center;justify-content:space-between}
.diag-label{font-size:.72rem;letter-spacing:.07em;text-transform:uppercase;opacity:.85;margin-bottom:4px}
.diag-name{font-family:'Orbitron',sans-serif;font-size:1.2rem;font-weight:800}
.diag-conf{font-family:var(--mono);font-size:1.5rem;font-weight:600}
.diag-sub{font-size:.72rem;opacity:.75}
.risk-badge{padding:5px 14px;border-radius:20px;font-size:.75rem;font-weight:700;
  letter-spacing:.06em;border:2px solid currentColor}
.risk-badge.High{color:var(--red);background:rgba(239,68,68,.15)}
.risk-badge.Low{color:var(--green);background:rgba(34,197,94,.15)}

/* ── Confidence breakdown ── */
.score-row{margin-bottom:12px}
.score-meta{display:flex;justify-content:space-between;margin-bottom:5px}
.score-name{font-size:.82rem;color:var(--text2)}
.score-pct{font-family:var(--mono);font-size:.82rem;font-weight:600}
.score-bar{height:6px;background:rgba(255,255,255,.06);border-radius:3px;overflow:hidden}
.score-fill{height:100%;border-radius:3px;transition:width .8s cubic-bezier(.25,.46,.45,.94)}

/* ── Description block ── */
.desc-block{background:var(--card2);border-radius:var(--r2);padding:16px;margin-bottom:16px;
  border-left:3px solid var(--teal)}
.desc-block h4{font-size:.78rem;letter-spacing:.07em;text-transform:uppercase;color:var(--teal);margin-bottom:8px}
.desc-block p{font-size:.85rem;color:var(--text2);line-height:1.65}

/* ── PDF button ── */
.btn-pdf{width:100%;padding:11px;border-radius:var(--r2);border:1px solid var(--border2);
  background:transparent;color:var(--text);font-family:var(--font);font-size:.88rem;
  cursor:pointer;transition:all .2s;display:flex;align-items:center;justify-content:center;gap:8px}
.btn-pdf:hover{background:var(--card2);border-color:var(--teal);color:var(--teal)}

/* ── History table ── */
.table-wrap{background:var(--card);border:1px solid var(--border);border-radius:var(--r);overflow:hidden}
.table-header{padding:18px 22px;border-bottom:1px solid var(--border);
  display:flex;align-items:center;justify-content:space-between}
.search-box{background:var(--card2);border:1px solid var(--border2);border-radius:var(--r2);
  padding:8px 14px;color:var(--text);font-family:var(--font);font-size:.85rem;outline:none;
  width:260px;transition:border-color .2s}
.search-box:focus{border-color:var(--teal)}
table{width:100%;border-collapse:collapse}
th{padding:11px 18px;text-align:left;font-size:.72rem;letter-spacing:.06em;
  text-transform:uppercase;color:var(--text3);border-bottom:1px solid var(--border);font-weight:600}
td{padding:13px 18px;font-size:.85rem;border-bottom:1px solid rgba(30,48,80,.5)}
tr:last-child td{border-bottom:none}
tr:hover td{background:rgba(255,255,255,.02)}
.pred-chip{padding:3px 10px;border-radius:12px;font-size:.73rem;font-weight:600;
  font-family:var(--mono);display:inline-block}
.conf-cell{font-family:var(--mono);color:var(--teal)}
.action-btn{padding:5px 12px;border-radius:6px;border:1px solid var(--border2);
  background:transparent;color:var(--text2);font-size:.76rem;cursor:pointer;
  transition:all .2s;font-family:var(--font)}
.action-btn:hover{border-color:var(--teal);color:var(--teal);background:var(--teal-glow)}

/* ── Loading overlay ── */
.loading{display:none;position:fixed;inset:0;background:rgba(8,13,20,.75);
  backdrop-filter:blur(4px);z-index:999;justify-content:center;align-items:center;
  flex-direction:column;gap:16px}
.loading.show{display:flex}
.spinner{width:44px;height:44px;border:3px solid var(--border2);
  border-top-color:var(--teal);border-radius:50%;animation:spin .8s linear infinite}
@keyframes spin{to{transform:rotate(360deg)}}
.loading-text{color:var(--text2);font-size:.9rem;font-family:var(--mono)}

/* ── Scan line animation on analyse btn ── */
@keyframes scanline{0%{top:-2px}100%{top:102%}}
.btn-analyse .sl{position:absolute;left:0;right:0;height:2px;
  background:rgba(255,255,255,.3);pointer-events:none;animation:scanline .9s linear infinite}

/* ── Toasts ── */
.toast-wrap{position:fixed;bottom:28px;right:28px;z-index:1000;display:flex;flex-direction:column;gap:8px}
.toast{background:var(--card);border:1px solid var(--border2);border-radius:var(--r2);
  padding:12px 18px;font-size:.85rem;color:var(--text);min-width:260px;
  animation:slideUp .3s ease;box-shadow:0 8px 24px rgba(0,0,0,.4)}
.toast.err{border-left:3px solid var(--red)}
.toast.ok{border-left:3px solid var(--green)}
@keyframes slideUp{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:translateY(0)}}

/* ── No data ── */
.empty{text-align:center;padding:60px 20px;color:var(--text3)}
.empty .ei{font-size:3rem;margin-bottom:12px}
</style>
</head>
<body>

<div class="loading" id="loading">
  <div class="spinner"></div>
  <div class="loading-text" id="loading-text">Analysing histology image…</div>
</div>

<div class="toast-wrap" id="toasts"></div>

<div class="shell">

  <!-- ── Sidebar ── -->
  <aside class="sidebar">
    <div class="logo">
      <div class="logo-title">ONCOSCAN AI</div>
      <div class="logo-sub">Lung Cancer Detection Platform</div>
    </div>
    <nav class="nav">
      <div class="nav-item active" data-view="dashboard" onclick="switchView('dashboard',this)">
        <span class="nav-icon">⬡</span> Dashboard
      </div>
      <div class="nav-item" data-view="analyze" onclick="switchView('analyze',this)">
        <span class="nav-icon">🔬</span> Analyze Image
      </div>
      <div class="nav-item" data-view="history" onclick="switchView('history',this)">
        <span class="nav-icon">📋</span> Scan History
        <span class="sidebar-badge" id="hist-badge">0</span>
      </div>
    </nav>
    <div style="padding:0 16px">
      <div style="background:var(--card2);border:1px solid var(--border);border-radius:10px;padding:14px 16px">
        <div style="font-size:.7rem;color:var(--text3);letter-spacing:.06em;text-transform:uppercase;margin-bottom:8px">System Status</div>
        <div style="display:flex;align-items:center;gap:8px;font-size:.82rem">
          <span style="width:8px;height:8px;border-radius:50%;background:var(--green);
            box-shadow:0 0 6px var(--green);display:inline-block"></span>
          VGG16 Model Active
        </div>
        <div style="display:flex;align-items:center;gap:8px;font-size:.82rem;margin-top:6px">
          <span style="width:8px;height:8px;border-radius:50%;background:var(--teal);
            box-shadow:0 0 6px var(--teal);display:inline-block"></span>
          SQLite Database Live
        </div>
      </div>
    </div>
  </aside>

  <!-- ── Main ── -->
  <main class="main">
    <div class="page-header" id="page-header">
      <div class="page-title" id="page-title">Dashboard</div>
      <div class="page-sub" id="page-sub">Platform overview and scan analytics</div>
    </div>

    <!-- ── DASHBOARD VIEW ── -->
    <div class="view active" id="view-dashboard">

      <div class="stats-grid" id="stats-grid">
        <div class="stat-card teal">
          <div class="stat-label">Total Scans</div>
          <div class="stat-value" id="st-total">—</div>
          <div class="stat-sub">All time</div>
          <div class="stat-icon">🗂️</div>
        </div>
        <div class="stat-card red">
          <div class="stat-label">Cancer Positive</div>
          <div class="stat-value" id="st-cancer">—</div>
          <div class="stat-sub" id="st-cancer-pct">—% of total</div>
          <div class="stat-icon">⚠️</div>
        </div>
        <div class="stat-card green">
          <div class="stat-label">Avg Confidence</div>
          <div class="stat-value" id="st-conf">—</div>
          <div class="stat-sub">% model certainty</div>
          <div class="stat-icon">📊</div>
        </div>
        <div class="stat-card purple">
          <div class="stat-label">Today's Scans</div>
          <div class="stat-value" id="st-today">—</div>
          <div class="stat-sub" id="st-date">—</div>
          <div class="stat-icon">📅</div>
        </div>
      </div>

      <div class="charts-row">
        <div class="chart-card">
          <div class="chart-title"><span></span>Class Distribution</div>
          <div class="chart-wrap"><canvas id="chart-dist"></canvas></div>
        </div>
        <div class="chart-card">
          <div class="chart-title"><span></span>7-Day Scan Trend</div>
          <div class="chart-wrap"><canvas id="chart-trend"></canvas></div>
        </div>
      </div>

      <div class="chart-card" style="margin-bottom:0">
        <div class="chart-title"><span></span>Average Confidence by Class</div>
        <div class="chart-wrap" style="height:180px"><canvas id="chart-conf-class"></canvas></div>
      </div>

    </div><!-- /dashboard -->

    <!-- ── ANALYZE VIEW ── -->
    <div class="view" id="view-analyze">
      <div class="analyze-grid">

        <!-- Left: patient + upload -->
        <div>
          <div class="panel" style="margin-bottom:16px">
            <div class="panel-title">Patient Information</div>
            <div class="field">
              <label>Full Name</label>
              <input type="text" id="f-name" placeholder="e.g. Ramesh Kumar" />
            </div>
            <div class="field">
              <label>Age</label>
              <input type="number" id="f-age" min="1" max="120" placeholder="Age in years" />
            </div>
            <div class="field">
              <label>Gender</label>
              <select id="f-gender">
                <option value="Unspecified">Unspecified</option>
                <option value="Male">Male</option>
                <option value="Female">Female</option>
                <option value="Other">Other</option>
              </select>
            </div>
          </div>

          <div class="panel">
            <div class="panel-title">Histology Image</div>
            <div class="upload-zone" id="upload-zone">
              <input type="file" id="file-input" accept=".png,.jpg,.jpeg" />
              <div class="upload-icon">🔬</div>
              <div class="upload-text"><strong>Click or drag</strong> to upload<br/>PNG · JPG · JPEG</div>
            </div>
            <img id="preview" class="preview-img" alt="Preview" />
            <button class="btn-analyse" id="btn-analyse" disabled onclick="runAnalysis()" style="position:relative;overflow:hidden">
              Run Analysis
            </button>
          </div>
        </div>

        <!-- Right: results -->
        <div>
          <!-- placeholder -->
          <div id="result-placeholder" class="panel" style="min-height:420px;display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center">
            <div style="font-size:3.5rem;margin-bottom:16px;opacity:.25">🩻</div>
            <div style="color:var(--text3);font-size:.92rem">Upload a histology image to<br/>receive an AI-assisted diagnosis</div>
          </div>

          <!-- actual result -->
          <div id="result-panel">
            <div class="panel" style="margin-bottom:16px">
              <div id="diagnosis-banner" class="diagnosis-banner">
                <div>
                  <div class="diag-label">Diagnosis</div>
                  <div class="diag-name" id="res-name">—</div>
                  <div class="diag-sub" id="res-icd">ICD-10: —</div>
                </div>
                <div style="text-align:right">
                  <div class="diag-conf" id="res-conf">—%</div>
                  <div class="diag-sub">confidence</div>
                  <div style="margin-top:6px">
                    <span class="risk-badge" id="res-risk">—</span>
                  </div>
                </div>
              </div>

              <div class="panel-title">Confidence Scores</div>
              <div id="score-bars"></div>
            </div>

            <div class="panel" style="margin-bottom:16px">
              <div class="desc-block">
                <h4>Clinical Description</h4>
                <p id="res-desc">—</p>
              </div>
              <div class="desc-block" style="border-color:var(--amber)">
                <h4 style="color:var(--amber)">Recommendations</h4>
                <p id="res-rec">—</p>
              </div>
            </div>

            <div class="panel">
              <button class="btn-pdf" id="btn-pdf" onclick="downloadPDF()">
                <span>⬇</span> Download Clinical PDF Report
              </button>
            </div>
          </div>
        </div>

      </div>
    </div><!-- /analyze -->

    <!-- ── HISTORY VIEW ── -->
    <div class="view" id="view-history">
      <div class="table-wrap">
        <div class="table-header">
          <div style="font-size:.88rem;font-weight:600;color:var(--text2)">All Scan Records</div>
          <input class="search-box" id="hist-search" placeholder="Search patient / diagnosis…" oninput="filterHistory()" />
        </div>
        <div id="hist-table-body"></div>
      </div>
    </div>

  </main>
</div>

<script>
// ── State ──────────────────────────────────────────────────────────────────
let currentScanId = null;
let historyData   = [];
let charts        = {};

// ── View switch ────────────────────────────────────────────────────────────
const VIEW_META = {
  dashboard: { title:'Dashboard', sub:'Platform overview and scan analytics' },
  analyze:   { title:'Analyze Image', sub:'Upload a histology sample for AI diagnosis' },
  history:   { title:'Scan History', sub:'All patient scan records' },
};
function switchView(name, el){
  document.querySelectorAll('.nav-item').forEach(n=>n.classList.remove('active'));
  el.classList.add('active');
  document.querySelectorAll('.view').forEach(v=>v.classList.remove('active'));
  document.getElementById('view-'+name).classList.add('active');
  document.getElementById('page-title').textContent = VIEW_META[name].title;
  document.getElementById('page-sub').textContent   = VIEW_META[name].sub;
  if(name==='dashboard') loadDashboard();
  if(name==='history')   loadHistory();
}

// ── Toast ──────────────────────────────────────────────────────────────────
function toast(msg, ok=true){
  const t=document.createElement('div');
  t.className='toast '+(ok?'ok':'err');
  t.textContent=msg;
  document.getElementById('toasts').appendChild(t);
  setTimeout(()=>t.remove(),3200);
}

// ── File upload / preview ──────────────────────────────────────────────────
document.getElementById('file-input').addEventListener('change',e=>{
  const f=e.target.files[0]; if(!f) return;
  const reader=new FileReader();
  reader.onload=ev=>{
    const img=document.getElementById('preview');
    img.src=ev.target.result; img.style.display='block';
  };
  reader.readAsDataURL(f);
  document.getElementById('btn-analyse').disabled=false;
  currentScanId=null;
  document.getElementById('result-panel').style.display='none';
  document.getElementById('result-placeholder').style.display='flex';
});
const uz=document.getElementById('upload-zone');
uz.addEventListener('dragover',e=>{e.preventDefault();uz.classList.add('drag')});
uz.addEventListener('dragleave',()=>uz.classList.remove('drag'));
uz.addEventListener('drop',e=>{
  e.preventDefault(); uz.classList.remove('drag');
  const f=e.dataTransfer.files[0]; if(!f) return;
  const dt=new DataTransfer(); dt.items.add(f);
  document.getElementById('file-input').files=dt.files;
  document.getElementById('file-input').dispatchEvent(new Event('change'));
});

// ── Run analysis ───────────────────────────────────────────────────────────
async function runAnalysis(){
  const file=document.getElementById('file-input').files[0];
  if(!file){ toast('Please select an image first.',false); return; }
  const fd=new FormData();
  fd.append('file',file);
  fd.append('patient_name', document.getElementById('f-name').value||'Anonymous');
  fd.append('patient_age',  document.getElementById('f-age').value||'N/A');
  fd.append('patient_gender', document.getElementById('f-gender').value);

  setLoading(true,'Analysing histology image…');
  try{
    const r=await fetch('/predict',{method:'POST',body:fd});
    const d=await r.json();
    if(d.error){ toast(d.error,false); return; }
    renderResult(d);
    toast('Analysis complete — '+d.prediction);
  }catch(e){
    toast('Network error: '+e.message,false);
  }finally{ setLoading(false); }
}

// ── Render result ──────────────────────────────────────────────────────────
function renderResult(d){
  currentScanId=d.scan_id;
  document.getElementById('result-placeholder').style.display='none';
  document.getElementById('result-panel').style.display='block';

  // Banner
  const banner=document.getElementById('diagnosis-banner');
  banner.style.background=d.hex+'22';
  banner.style.border='1px solid '+d.hex+'55';
  document.getElementById('res-name').textContent=d.prediction;
  document.getElementById('res-name').style.color=d.hex;
  document.getElementById('res-icd').textContent='ICD-10: '+d.icd10;
  document.getElementById('res-conf').textContent=parseFloat(d.confidence).toFixed(1)+'%';
  const rb=document.getElementById('res-risk');
  rb.textContent=d.risk+' Risk'; rb.className='risk-badge '+d.risk;

  // Bars
  const CLASS_COLORS=['#ef4444','#f97316','#22c55e','#a855f7'];
  const CLASS_NAMES=['Adenocarcinoma','Large Cell Carcinoma','Normal','Squamous Cell Carcinoma'];
  const sb=document.getElementById('score-bars'); sb.innerHTML='';
  CLASS_NAMES.forEach((name,i)=>{
    const pct=parseFloat(d.all_scores[name]||0).toFixed(1);
    sb.innerHTML+=`
      <div class="score-row">
        <div class="score-meta">
          <span class="score-name">${name}</span>
          <span class="score-pct" style="color:${CLASS_COLORS[i]}">${pct}%</span>
        </div>
        <div class="score-bar">
          <div class="score-fill" style="width:${pct}%;background:${CLASS_COLORS[i]}"></div>
        </div>
      </div>`;
  });

  document.getElementById('res-desc').textContent=d.description;
  document.getElementById('res-rec').textContent=d.recommendation;
}

function downloadPDF(){
  if(!currentScanId){ toast('No scan to export.',false); return; }
  window.open('/report/pdf/'+currentScanId,'_blank');
}

// ── Dashboard ──────────────────────────────────────────────────────────────
async function loadDashboard(){
  setLoading(true,'Loading analytics…');
  try{
    const r=await fetch('/api/stats'); const d=await r.json();
    document.getElementById('st-total').textContent=d.total;
    document.getElementById('st-cancer').textContent=d.cancer_positive;
    document.getElementById('st-cancer-pct').textContent=
      d.total>0?(((d.cancer_positive/d.total)*100).toFixed(1)+'% of total'):'—';
    document.getElementById('st-conf').textContent=d.avg_confidence+'%';
    document.getElementById('st-today').textContent=d.today;
    document.getElementById('st-date').textContent=new Date().toLocaleDateString('en-IN',{day:'numeric',month:'short',year:'numeric'});
    document.getElementById('hist-badge').textContent=d.total;
    renderCharts(d);
  }catch(e){ toast('Failed to load stats.',false); }
  finally{ setLoading(false); }
}

function renderCharts(d){
  const FONT='Exo 2'; const TEXT='#8aa4c8'; const GRID='rgba(30,48,80,.5)';
  Chart.defaults.color=TEXT; Chart.defaults.font.family=FONT;

  // Distribution doughnut
  if(charts.dist) charts.dist.destroy();
  const distLabels=Object.keys(d.distribution);
  const distVals=Object.values(d.distribution);
  const distColors=['#ef4444','#f97316','#22c55e','#a855f7'];
  charts.dist=new Chart(document.getElementById('chart-dist'),{
    type:'doughnut',
    data:{labels:distLabels,datasets:[{data:distVals,
      backgroundColor:distColors.map(c=>c+'cc'),borderColor:distColors,borderWidth:2,
      hoverOffset:8}]},
    options:{responsive:true,maintainAspectRatio:false,cutout:'68%',
      plugins:{legend:{position:'right',labels:{boxWidth:12,padding:14,font:{size:11}}},
        tooltip:{callbacks:{label:c=>` ${c.label}: ${c.raw} scans`}}}}
  });

  // Trend line
  if(charts.trend) charts.trend.destroy();
  const tLabels=d.trend.map(t=>t.date.slice(5));
  const tVals=d.trend.map(t=>t.count);
  charts.trend=new Chart(document.getElementById('chart-trend'),{
    type:'line',
    data:{labels:tLabels,datasets:[{label:'Scans',data:tVals,fill:true,
      borderColor:'#14b8a6',backgroundColor:'rgba(20,184,166,.1)',tension:.45,
      pointBackgroundColor:'#14b8a6',pointRadius:4,borderWidth:2}]},
    options:{responsive:true,maintainAspectRatio:false,
      scales:{x:{grid:{color:GRID}},y:{grid:{color:GRID},beginAtZero:true,ticks:{precision:0}}},
      plugins:{legend:{display:false}}}
  });

  // Confidence per class bar
  if(charts.confClass) charts.confClass.destroy();
  const ccLabels=Object.keys(d.conf_per_class);
  const ccVals=Object.values(d.conf_per_class);
  charts.confClass=new Chart(document.getElementById('chart-conf-class'),{
    type:'bar',
    data:{labels:ccLabels,datasets:[{label:'Avg Confidence %',data:ccVals,
      backgroundColor:['#ef4444cc','#f97316cc','#22c55ecc','#a855f7cc'],
      borderColor:['#ef4444','#f97316','#22c55e','#a855f7'],borderWidth:1.5,borderRadius:6}]},
    options:{responsive:true,maintainAspectRatio:false,indexAxis:'y',
      scales:{x:{grid:{color:GRID},min:0,max:100,ticks:{callback:v=>v+'%'}},
               y:{grid:{color:GRID}}},
      plugins:{legend:{display:false}}}
  });
}

// ── History ────────────────────────────────────────────────────────────────
async function loadHistory(){
  setLoading(true,'Loading records…');
  try{
    const r=await fetch('/api/history'); historyData=await r.json();
    document.getElementById('hist-badge').textContent=historyData.length;
    renderTable(historyData);
  }catch(e){ toast('Failed to load history.',false); }
  finally{ setLoading(false); }
}

function filterHistory(){
  const q=document.getElementById('hist-search').value.toLowerCase();
  renderTable(historyData.filter(r=>
    r.patient_name.toLowerCase().includes(q)||
    r.prediction.toLowerCase().includes(q)
  ));
}

function renderTable(rows){
  const COLORS={
    'Adenocarcinoma':'background:#ef444420;color:#ef4444',
    'Large Cell Carcinoma':'background:#f9731620;color:#f97316',
    'Normal':'background:#22c55e20;color:#22c55e',
    'Squamous Cell Carcinoma':'background:#a855f720;color:#a855f7',
  };
  const tb=document.getElementById('hist-table-body');
  if(!rows.length){
    tb.innerHTML='<div class="empty"><div class="ei">📂</div><div>No scan records found</div></div>';
    return;
  }
  tb.innerHTML=`<table>
    <thead><tr>
      <th>#</th><th>Date / Time</th><th>Patient</th><th>Age</th>
      <th>Gender</th><th>Prediction</th><th>Confidence</th><th>Actions</th>
    </tr></thead>
    <tbody>${rows.map((r,i)=>`
      <tr>
        <td style="color:var(--text3);font-family:var(--mono)">${i+1}</td>
        <td style="font-family:var(--mono);font-size:.78rem;color:var(--text2)">${r.created_at}</td>
        <td style="font-weight:500">${r.patient_name}</td>
        <td>${r.patient_age||'—'}</td>
        <td style="color:var(--text2)">${r.patient_gender||'—'}</td>
        <td><span class="pred-chip" style="${COLORS[r.prediction]||''}">${r.prediction}</span></td>
        <td class="conf-cell">${parseFloat(r.confidence).toFixed(1)}%</td>
        <td>
          <button class="action-btn" onclick="window.open('/report/pdf/${r.id}','_blank')">
            ⬇ PDF
          </button>
        </td>
      </tr>`).join('')}
    </tbody>
  </table>`;
}

// ── Loading ────────────────────────────────────────────────────────────────
function setLoading(show, msg=''){
  const el=document.getElementById('loading');
  el.classList.toggle('show',show);
  if(msg) document.getElementById('loading-text').textContent=msg;
}

// ── Init ───────────────────────────────────────────────────────────────────
window.addEventListener('DOMContentLoaded',loadDashboard);
</script>
</body>
</html>"""

if __name__ == "__main__":
    init_db()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000)
    print(f"[OncoScan AI] Public URL → {public_url}")
    app.run(port=5000)

[OncoScan AI] Loading VGG16 model …


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


[OncoScan AI] Model ready.
[OncoScan AI] Public URL → NgrokTunnel: "https://2a5d-34-86-59-199.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 15:25:47] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 15:25:49] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 15:25:49] "GET /api/stats HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 15:25:53] "GET /api/history HTTP/1.1" 200 -
/tmp/ipykernel_1092/855923593.py:182: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "ONCOSCAN AI", ln=True)
/tmp/ipykernel_1092/855923593.py:190: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=False use new_x=XPos.RIGHT, new_y=YPos.TOP.
  pdf.cell(90, 7, f"Patient: {patient_name}", ln=False)
/tmp/ipykernel_1092/855923593.py:191: Depre